# The KV Cache: How Transformers Avoid Recomputing the Past

The runnable code of [The KV Cache: How Transformers Avoid Recomputing the Past](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/kv-cache-explained.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "kv-cache-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Cached and uncached give identical results

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/kv-cache-explained/#cached-and-uncached-give-identical-results)

In [ ]:
import numpy as np

def attention(Q, K, V):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))
    w /= w.sum(axis=-1, keepdims=True)
    return w @ V

def fmt(v):
    return "[" + " ".join(f"{x:.3f}" for x in v) + "]"

W_Q = np.array([[1, 0], [1, 1]], dtype=float)
W_K = np.array([[0, 1], [1, 0]], dtype=float)
W_V = np.array([[1, 1], [0, 1]], dtype=float)
X = np.array([[1, 0], [0, 1], [1, -1], [0.5, 0.5]], dtype=float)

n = len(X)
K_cache, V_cache = [], []
naive_rows = cached_rows = 0

for t in range(1, n + 1):
    # naive: recompute K and V for all t tokens from scratch, every step
    Kt, Vt = X[:t] @ W_K, X[:t] @ W_V
    y_naive = attention(X[t - 1:t] @ W_Q, Kt, Vt)[0]
    naive_rows += t

    # cached: compute K and V for only the new token, append to the cache
    K_cache.append(X[t - 1] @ W_K)
    V_cache.append(X[t - 1] @ W_V)
    y_cached = attention(X[t - 1:t] @ W_Q, np.array(K_cache), np.array(V_cache))[0]
    cached_rows += 1

    print(f"step {t}  no-cache={fmt(y_naive)}  cached={fmt(y_cached)}  rows: {t} vs 1")

print(f"total K/V rows computed: {naive_rows} (= n(n+1)/2)  vs  {cached_rows} (= n)")

The post printed:

```text
step 1  no-cache=[1.000 1.000]  cached=[1.000 1.000]  rows: 1 vs 1
step 2  no-cache=[0.500 1.000]  cached=[0.500 1.000]  rows: 2 vs 1
step 3  no-cache=[0.497 0.752]  cached=[0.497 0.752]  rows: 3 vs 1
step 4  no-cache=[0.508 0.880]  cached=[0.508 0.880]  rows: 4 vs 1
total K/V rows computed: 10 (= n(n+1)/2)  vs  4 (= n)
```